# Splash Attention vs Flash Attention

Google's Splash Attention is a TPU-native sparse attention implementation. This notebook compares design assumptions and identifies where the tradeoffs diverge.

## What is Splash Attention?

Splash Attention (`jax/experimental/pallas/ops/tpu/splash_attention/`) implements *sparse* attention with a block-diagonal mask structure. Rather than attending to every key-value pair, each query attends to a structured subset.

Supported mask types:
- **Causal (local + causal)**: sliding window with a causal constraint
- **Full**: standard dense attention (equivalent to flash attention)
- **Local**: sliding window (each query attends to a fixed radius of keys)
- **BCSR**: block compressed sparse row — arbitrary sparse patterns

The key design decision: the sparsity pattern is fixed at kernel compile time and encoded in the block structure. This is different from runtime masking.

## Design comparison

| Dimension | Flash Attention (ours) | Splash Attention |
|---|---|---|
| Sparsity | Dense | Structured sparse |
| Mask | Causal or none (runtime) | Block pattern (compile time) |
| TPU optimization | VMEM tiling, online softmax | Block-sparse skip, DMA coalescing |
| Memory savings | O(n²) → O(n) via no materialization | O(n²) → O(n * sparsity) via skip |
| Backward pass | Recomputation | Recomputation |
| Primary use case | Long dense context | Sliding window, structured LLM attention |
| GQA support | Yes (this project) | Yes (built-in) |

## Where the intuitions diverge

**Flash Attention** was designed to maximize HBM bandwidth efficiency for dense attention. Its innovation is the tiling structure that keeps VMEM utilization high across the full KV sequence.

**Splash Attention** exploits the observation that in practice, many LLMs use structured sparse patterns (local + global attention, sliding window, etc.). By encoding the sparsity in the block structure, it skips entire KV tile loads from HBM — not just avoiding the matmul, but the DMA transfer too.

For dense full attention at long sequences, both approaches are memory-efficient. Splash's advantage appears when sparsity > ~50% — the skipped DMA transfers become the dominant benefit.

In [ ]:
# Side-by-side: flash attention (our impl) vs Splash Attention API
# Run this on a TPU runtime with JAX >= 0.4.25

import jax
import jax.numpy as jnp
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent / "03_pallas_kernels"))
from flash_fwd import flash_attention_forward
from utils import get_block_sizes

try:
    from jax.experimental.pallas.ops.tpu.splash_attention import (
        splash_attention_kernel,
        splash_attention_mask,
    )
    SPLASH_AVAILABLE = True
except ImportError:
    print("Splash Attention not available in this JAX version.")
    SPLASH_AVAILABLE = False

BATCH, HEADS, SEQ, D = 1, 4, 2048, 128
key = jax.random.PRNGKey(0)
k1, k2, k3 = jax.random.split(key, 3)
shape = (BATCH, HEADS, SEQ, D)
dtype = jnp.bfloat16

q = jax.random.normal(k1, shape, dtype=dtype)
k = jax.random.normal(k2, shape, dtype=dtype)
v = jax.random.normal(k3, shape, dtype=dtype)

# Flash attention forward
block_sizes = get_block_sizes(SEQ, D, dtype)
o_flash, _, _ = flash_attention_forward(q, k, v, causal=True, block_sizes=block_sizes)
print(f"Flash output shape: {o_flash.shape}, dtype: {o_flash.dtype}")

In [ ]:
if SPLASH_AVAILABLE:
    # Splash attention with causal mask
    # The mask is specified as a structured object, not a boolean array.
    # 'CausalMask' is equivalent to our causal=True.
    mask = splash_attention_mask.CausalMask(shape=(SEQ, SEQ))

    # Splash attention uses (batch, heads, seq, d) layout, same as ours.
    # The API returns only the output (not m, l).
    o_splash = splash_attention_kernel.splash_attention(
        q, k, v,
        mask=mask,
        block_sizes=splash_attention_kernel.BlockSizes(
            block_q=block_sizes.block_q,
            block_kv=block_sizes.block_kv,
        ),
    )

    # Compare outputs — should match within bfloat16 tolerance
    import numpy as np
    max_diff = float(jnp.abs(o_flash - o_splash).max())
    print(f"Max absolute difference (flash vs splash): {max_diff:.4f}")
    print(f"Expected: < 0.05 for bfloat16")
else:
    print("Skipping Splash Attention comparison (not available).")
    print("To run: use JAX >= 0.4.25 on a TPU runtime.")

## When to use which

**Use Flash Attention (this impl) when:**
- Dense full attention with optional causal masking
- Need a clean, documented, modifiable kernel
- GQA with a simple head-mapping change

**Use Splash Attention when:**
- Sliding window or other structured sparse patterns
- Production deployment on Google TPUs where the library is maintained
- Very long sequences where sparsity matters (> 100K tokens)

## What TPU silicon rewards (and GPU silicon doesn't)

The key difference this project exposes:

**GPU Flash Attention** is primarily an *HBM bandwidth* optimization. The SRAM on a GPU is a cache — the programmer doesn't directly control what lives there. Flash Attention works by reordering computation to be cache-friendly.

**TPU Pallas kernels** use VMEM as an explicit *scratchpad*. You control exactly what moves between HBM and VMEM. Splash Attention can skip DMA transfers for masked tiles entirely — something that's harder to express in CUDA's cache model.

This is the fundamental hardware divergence: TPU rewards explicit memory management; GPU rewards cache-friendly access patterns. The same algorithm (Flash Attention) adapts differently to each.